# HW08-09: PyTorch MLP

## 1) Импорты, seed и устройство

In [1]:
import json
import random
from copy import deepcopy
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

SEED = 42


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"torch: {torch.__version__}")


Device: cpu
torch: 2.10.0+cpu


## 2) Данные и DataLoader

In [2]:
ROOT = Path(".")
ARTIFACTS_DIR = ROOT / "artifacts"
FIGURES_DIR = ARTIFACTS_DIR / "figures"
DATA_DIR = ROOT / "data"

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 256
VAL_RATIO = 0.2
DATASET_NAME = "EMNIST"

transform = transforms.ToTensor()

train_full = datasets.EMNIST(
    root=str(DATA_DIR),
    split="balanced",
    train=True,
    download=True,
    transform=transform,
)

test_dataset = datasets.EMNIST(
    root=str(DATA_DIR),
    split="balanced",
    train=False,
    download=True,
    transform=transform,
)

print(f"{DATASET_NAME} (balanced) loaded from torchvision.")

train_size = int((1 - VAL_RATIO) * len(train_full))
val_size = len(train_full) - train_size
split_gen = torch.Generator().manual_seed(SEED)

train_dataset, val_dataset = random_split(
    train_full,
    [train_size, val_size],
    generator=split_gen,
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Train size: {len(train_dataset)}")
print(f"Val size:   {len(val_dataset)}")
print(f"Test size:  {len(test_dataset)}")

x, y = next(iter(train_loader))
print(f"x.shape: {x.shape}")
print(f"y.shape: {y.shape}")
print(f"x range: [{x.min().item():.3f}, {x.max().item():.3f}]")
print(f"labels min/max: ({int(y.min())}, {int(y.max())})")

EMNIST (balanced) loaded from torchvision.
Train size: 90240
Val size:   22560
Test size:  18800
x.shape: torch.Size([256, 1, 28, 28])
y.shape: torch.Size([256])
x range: [0.000, 1.000]
labels min/max: (0, 46)


## 3) Модель MLP и цикл обучения

In [3]:
class MLP(nn.Module):
    def __init__(
        self,
        input_size: int,
        num_classes: int,
        hidden_sizes: list[int],
        activation: str = "relu",
        dropout: float = 0.0,
        use_batchnorm: bool = False,
    ):
        super().__init__()

        if activation.lower() != "relu":
            raise ValueError("Сейчас поддерживается только activation='relu'.")

        layers = [nn.Flatten()]
        prev_size = input_size

        for h in hidden_sizes:
            layers.append(nn.Linear(prev_size, h))
            if use_batchnorm:
                layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev_size = h

        layers.append(nn.Linear(prev_size, num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader, criterion: nn.Module, device: torch.device) -> dict:
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total = 0

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        logits = model(xb)
        loss = criterion(logits, yb)

        total_loss += loss.item() * xb.size(0)
        preds = logits.argmax(dim=1)
        total_correct += (preds == yb).sum().item()
        total += xb.size(0)

    return {
        "loss": total_loss / total,
        "accuracy": total_correct / total,
    }


def train_one_epoch(model: nn.Module, loader: DataLoader, criterion: nn.Module, optimizer, device: torch.device) -> dict:
    model.train()
    total_loss = 0.0
    total_correct = 0
    total = 0

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * xb.size(0)
        preds = logits.argmax(dim=1)
        total_correct += (preds == yb).sum().item()
        total += xb.size(0)

    return {
        "loss": total_loss / total,
        "accuracy": total_correct / total,
    }


def make_model_summary(hidden_sizes, activation, dropout, use_batchnorm):
    return (
        f"hidden={hidden_sizes}; activation={activation}; "
        f"dropout={dropout}; batchnorm={use_batchnorm}"
    )


In [4]:
def run_experiment(
    experiment_id: str,
    model_cfg: dict,
    optimizer_name: str,
    lr: float,
    max_epochs: int,
    momentum: float = 0.0,
    weight_decay: float = 0.0,
    early_stopping_patience: int | None = None,
):
    model = MLP(**model_cfg).to(DEVICE)
    criterion = nn.CrossEntropyLoss()

    if optimizer_name.lower() == "adam":
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
        optimizer_label = "Adam"
    elif optimizer_name.lower() == "sgd":
        optimizer = torch.optim.SGD(
            model.parameters(),
            lr=lr,
            momentum=momentum,
            weight_decay=weight_decay,
        )
        optimizer_label = "SGD"
    else:
        raise ValueError("optimizer_name должен быть 'adam' или 'sgd'")

    history = {
        "epoch": [],
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": [],
    }

    best_state = None
    best_val_acc = -1.0
    best_val_loss = float("inf")
    patience_counter = 0

    for epoch in range(1, max_epochs + 1):
        train_metrics = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
        val_metrics = evaluate(model, val_loader, criterion, DEVICE)

        history["epoch"].append(epoch)
        history["train_loss"].append(train_metrics["loss"])
        history["train_acc"].append(train_metrics["accuracy"])
        history["val_loss"].append(val_metrics["loss"])
        history["val_acc"].append(val_metrics["accuracy"])

        if val_metrics["loss"] < best_val_loss:
            best_val_loss = val_metrics["loss"]

        if val_metrics["accuracy"] > best_val_acc:
            best_val_acc = val_metrics["accuracy"]
            best_state = deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if early_stopping_patience is not None and patience_counter >= early_stopping_patience:
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    row = {
        "experiment_id": experiment_id,
        "dataset": DATASET_NAME,
        "seed": SEED,
        "model_summary": make_model_summary(
            model_cfg["hidden_sizes"],
            model_cfg["activation"],
            model_cfg["dropout"],
            model_cfg["use_batchnorm"],
        ),
        "optimizer": optimizer_label,
        "lr": lr,
        "momentum": momentum if optimizer_label == "SGD" else 0.0,
        "weight_decay": weight_decay,
        "epochs_trained": len(history["epoch"]),
        "best_val_accuracy": best_val_acc,
        "best_val_loss": best_val_loss,
    }

    return model, history, row

## 4) Эксперименты, часть A (S08): E1-E4

In [5]:
INPUT_SIZE = 28 * 28
NUM_CLASSES = len(train_full.classes)

runs = []
histories = {}
models = {}

base_model_cfg = {
    "input_size": INPUT_SIZE,
    "num_classes": NUM_CLASSES,
    "hidden_sizes": [256, 128],
    "activation": "relu",
    "dropout": 0.0,
    "use_batchnorm": False,
}

# E1: base
model_e1, hist_e1, row_e1 = run_experiment(
    experiment_id="E1",
    model_cfg=base_model_cfg,
    optimizer_name="Adam",
    lr=1e-3,
    max_epochs=15,
)
runs.append(row_e1)
histories["E1"] = hist_e1
models["E1"] = model_e1
print("E1 done", row_e1["best_val_accuracy"])

# E2: Dropout
cfg_e2 = deepcopy(base_model_cfg)
cfg_e2["dropout"] = 0.3
model_e2, hist_e2, row_e2 = run_experiment(
    experiment_id="E2",
    model_cfg=cfg_e2,
    optimizer_name="Adam",
    lr=1e-3,
    max_epochs=15,
)
runs.append(row_e2)
histories["E2"] = hist_e2
models["E2"] = model_e2
print("E2 done", row_e2["best_val_accuracy"])

# E3: BatchNorm
cfg_e3 = deepcopy(base_model_cfg)
cfg_e3["use_batchnorm"] = True
model_e3, hist_e3, row_e3 = run_experiment(
    experiment_id="E3",
    model_cfg=cfg_e3,
    optimizer_name="Adam",
    lr=1e-3,
    max_epochs=15,
)
runs.append(row_e3)
histories["E3"] = hist_e3
models["E3"] = model_e3
print("E3 done", row_e3["best_val_accuracy"])

# Выбор лучшей модели между E2/E3 по val_accuracy
best_between_e2_e3 = max([row_e2, row_e3], key=lambda x: x["best_val_accuracy"])
selected_id = best_between_e2_e3["experiment_id"]
selected_cfg = deepcopy(cfg_e2 if selected_id == "E2" else cfg_e3)
print("Selected for E4:", selected_id)

# E4: лучшая из (E2/E3) + EarlyStopping
model_e4, hist_e4, row_e4 = run_experiment(
    experiment_id="E4",
    model_cfg=selected_cfg,
    optimizer_name="Adam",
    lr=1e-3,
    max_epochs=15,
    early_stopping_patience=5,
)
runs.append(row_e4)
histories["E4"] = hist_e4
models["E4"] = model_e4
print("E4 done", row_e4["best_val_accuracy"], "epochs:", row_e4["epochs_trained"])

E1 done 0.8450354609929078
E2 done 0.8434840425531915
E3 done 0.8498670212765957
Selected for E4: E3
E4 done 0.8463209219858157 epochs: 11


In [6]:
# Финальная оценка лучшей модели (E4) на test только один раз
criterion = nn.CrossEntropyLoss()
test_metrics = evaluate(models["E4"], test_loader, criterion, DEVICE)
print("E4 test accuracy:", test_metrics["accuracy"])
print("E4 test loss:", test_metrics["loss"])

# Сохранение best_model.pt и best_config.json
best_model_path = ARTIFACTS_DIR / "best_model.pt"
torch.save(models["E4"].state_dict(), best_model_path)

best_config = {
    "dataset": DATASET_NAME,
    "seed": SEED,
    "selected_from": selected_id,
    "final_experiment": "E4",
    "model_cfg": selected_cfg,
    "optimizer": "Adam",
    "lr": 1e-3,
    "weight_decay": 0.0,
    "early_stopping_patience": 5,
    "epochs_trained": row_e4["epochs_trained"],
    "best_val_accuracy": row_e4["best_val_accuracy"],
    "best_val_loss": row_e4["best_val_loss"],
    "test_accuracy": test_metrics["accuracy"],
    "test_loss": test_metrics["loss"],
}

with open(ARTIFACTS_DIR / "best_config.json", "w", encoding="utf-8") as f:
    json.dump(best_config, f, ensure_ascii=False, indent=2)

E4 test accuracy: 0.8395744680851064
E4 test loss: 0.48211185254949207


## 5) Эксперименты, часть B (S09): O1-O3

In [7]:
# Архитектура для O1-O3 фиксирована: та же, что в E4
opt_model_cfg = deepcopy(selected_cfg)

# O1: слишком большой LR
model_o1, hist_o1, row_o1 = run_experiment(
    experiment_id="O1",
    model_cfg=opt_model_cfg,
    optimizer_name="Adam",
    lr=1e-1,
    max_epochs=6,
)
runs.append(row_o1)
histories["O1"] = hist_o1
print("O1 done", row_o1["best_val_accuracy"])

# O2: слишком маленький LR
model_o2, hist_o2, row_o2 = run_experiment(
    experiment_id="O2",
    model_cfg=opt_model_cfg,
    optimizer_name="Adam",
    lr=1e-5,
    max_epochs=6,
)
runs.append(row_o2)
histories["O2"] = hist_o2
print("O2 done", row_o2["best_val_accuracy"])

# O3: SGD + momentum + weight_decay
model_o3, hist_o3, row_o3 = run_experiment(
    experiment_id="O3",
    model_cfg=opt_model_cfg,
    optimizer_name="SGD",
    lr=1e-2,
    momentum=0.9,
    weight_decay=1e-4,
    max_epochs=10,
)
runs.append(row_o3)
histories["O3"] = hist_o3
print("O3 done", row_o3["best_val_accuracy"])


O1 done 0.807136524822695
O2 done 0.5890070921985816
O3 done 0.8433510638297872


## 6) Сохранение `runs.csv`

In [8]:
runs_df = pd.DataFrame(runs)
order = ["E1", "E2", "E3", "E4", "O1", "O2", "O3"]
runs_df["experiment_id"] = pd.Categorical(runs_df["experiment_id"], categories=order, ordered=True)
runs_df = runs_df.sort_values("experiment_id").reset_index(drop=True)

runs_csv_path = ARTIFACTS_DIR / "runs.csv"
runs_df.to_csv(runs_csv_path, index=False)
runs_df

,experiment_id,dataset,seed,model_summary,optimizer,lr,momentum,weight_decay,epochs_trained,best_val_accuracy,best_val_loss
0,E1,EMNIST,42,"hidden=[256, 128]; activation=relu; dropout=0....",Adam,0.00100,0.0,0.0000,15,0.845035,0.480750
1,E2,EMNIST,42,"hidden=[256, 128]; activation=relu; dropout=0....",Adam,0.00100,0.0,0.0000,15,0.843484,0.472052
2,E3,EMNIST,42,"hidden=[256, 128]; activation=relu; dropout=0....",Adam,0.00100,0.0,0.0000,15,0.849867,0.467995
3,E4,EMNIST,42,"hidden=[256, 128]; activation=relu; dropout=0....",Adam,0.00100,0.0,0.0000,11,0.846321,0.465203
4,O1,EMNIST,42,"hidden=[256, 128]; activation=relu; dropout=0....",Adam,0.10000,0.0,0.0000,6,0.807137,0.610446
5,O2,EMNIST,42,"hidden=[256, 128]; activation=relu; dropout=0....",Adam,0.00001,0.0,0.0000,6,0.589007,2.274429
6,O3,EMNIST,42,"hidden=[256, 128]; activation=relu; dropout=0....",SGD,0.01000,0.9,0.0001,10,0.843351,0.474267


## 7) Графики для отчёта

In [9]:
import pickle

with open(ARTIFACTS_DIR / "histories.pkl", "wb") as f:
    pickle.dump(histories, f)

print("histories saved")

histories saved


In [ ]:
import pickle
from pathlib import Path
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

ROOT = Path(".")
ARTIFACTS_DIR = ROOT / "artifacts"
FIGURES_DIR = ARTIFACTS_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

with open(ARTIFACTS_DIR / "histories.pkl", "rb") as f:
    histories = pickle.load(f)

def plot_history_two_metrics(history, title, save_path):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(history["epoch"], history["train_loss"], label="train_loss")
    axes[0].plot(history["epoch"], history["val_loss"], label="val_loss")
    axes[0].set_title(f"{title}: Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    axes[1].plot(history["epoch"], history["train_acc"], label="train_acc")
    axes[1].plot(history["epoch"], history["val_acc"], label="val_acc")
    axes[1].set_title(f"{title}: Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy")
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    fig.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)

plot_history_two_metrics(
    histories["E4"],
    "E4 (best run)",
    FIGURES_DIR / "curves_best.png",
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(histories["O1"]["epoch"], histories["O1"]["train_loss"], label="O1 train_loss")
axes[0].plot(histories["O1"]["epoch"], histories["O1"]["val_loss"], label="O1 val_loss")
axes[0].set_title("O1: Adam lr=1e-1 (too large)")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(histories["O2"]["epoch"], histories["O2"]["train_loss"], label="O2 train_loss")
axes[1].plot(histories["O2"]["epoch"], histories["O2"]["val_loss"], label="O2 val_loss")
axes[1].set_title("O2: Adam lr=1e-5 (too small)")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid(alpha=0.3)

fig.tight_layout()
fig.savefig(FIGURES_DIR / "curves_lr_extremes.png", dpi=150, bbox_inches="tight")
plt.close(fig)

print("figures saved")

## 8) Проверка готовых артефактов

In [ ]:
expected = [
    ARTIFACTS_DIR / "runs.csv",
    ARTIFACTS_DIR / "best_model.pt",
    ARTIFACTS_DIR / "best_config.json",
    FIGURES_DIR / "curves_best.png",
    FIGURES_DIR / "curves_lr_extremes.png",
]

for p in expected:
    print(f"{p}: {'OK' if p.exists() else 'MISSING'}")


In [1]:
import pickle
from pathlib import Path

pkl_path = Path("artifacts") / "histories.pkl"

with open(pkl_path, "rb") as f:
    histories = pickle.load(f)

print(type(histories))
print(histories.keys())

for key in ["E4", "O1", "O2"]:
    print(f"\n{key}:")
    print(type(histories[key]))
    print(histories[key].keys())
    print("epochs:", len(histories[key]["epoch"]))
    print("train_loss first/last:", histories[key]["train_loss"][0], histories[key]["train_loss"][-1])
    print("val_loss first/last:", histories[key]["val_loss"][0], histories[key]["val_loss"][-1])

<class 'dict'>
dict_keys(['E1', 'E2', 'E3', 'E4', 'O1', 'O2', 'O3'])

E4:
<class 'dict'>
dict_keys(['epoch', 'train_loss', 'train_acc', 'val_loss', 'val_acc'])
epochs: 11
train_loss first/last: 1.1467414250610568 0.25192546819118744
val_loss first/last: 0.6428638959607333 0.4820454847305379

O1:
<class 'dict'>
dict_keys(['epoch', 'train_loss', 'train_acc', 'val_loss', 'val_acc'])
epochs: 6
train_loss first/last: 1.0392995082740242 0.5167979812791161
val_loss first/last: 0.8051459719103279 0.6128618670693526

O2:
<class 'dict'>
dict_keys(['epoch', 'train_loss', 'train_acc', 'val_loss', 'val_acc'])
epochs: 6
train_loss first/last: 3.579580940760619 2.337464827679573
val_loss first/last: 3.325563923517863 2.2744294464165438
